In [ ]:
%%bash
KEEP="pvitb1_imagenet_satt_tutelmoe_FINAL_FTs4RUN"
BASE="/workspace/ModelTraining/checkpoints"

for dir in "$BASE"/*/; do
    name=$(basename "$dir")
    if [ "$name" != "$KEEP" ]; then
        echo "Deleting: $dir"
        rm -rf "$dir"
    fi
done
echo "Done. Kept: $KEEP"


In [ ]:
import sys
import torch 
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from timm.models.registry import register_model ##REMOVE
from timm.models.vision_transformer import _cfg ##REMOVE
import math
from torchvision import datasets, transforms
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset, IterableDataset, Subset, random_split

from tqdm.notebook import tqdm

import numpy as np


from einops import rearrange
from einops.layers.torch import Rearrange

try:
  import pytorch_lightning as pl
except ModuleNotFoundError:
  !pip install --quiet pytorch-lightning>=1.5
  import pytorch_lightning as pl

try:
  from thop import profile
except ModuleNotFoundError:
  %pip install thop
  from thop import profile

try:
  from tutel import moe as tutel_moe
except ModuleNotFoundError:
  %pip install -v -U --no-build-isolation git+https://github.com/microsoft/tutel@main
  from tutel import moe as tutel_moe

  import pytorch_lightning as pl

from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger

try:
  import datasets
except ModuleNotFoundError:
  %pip install datasets
  import datasets

try:
  import huggingface_hub
except ModuleNotFoundError:
  %pip install --upgrade huggingface_hub
  import huggingface_hub

from huggingface_hub import login
from datasets import load_dataset
import os

In [ ]:
from functools import partial
NORM_LAYER = partial(nn.LayerNorm, eps=1e-6)
NORM_LAYER_STAGE4 = None   # None = same as stages 1-3
print("[Norm] LayerNorm throughout")

In [ ]:
config = {
    #Experiment Data
    "run_name": "pvt_tutel_LR_aux_fixed_p2_v10",
    "experiment_group": "model run",
    "version": 10,
    "ckpt_path": "/workspace/ModelTraining/checkpoints/pvt_tutel_LR_aux_fixed_v7/epoch=53-MulticlassAccuracy/val=0.7227.ckpt",
     #"/workspace/ModelTraining/checkpoints/pvitb1_imagenet_satt_tutelmoe_FINAL_FTs4RUN/last.ckpt", <- was the base FT backbone

    # Model Architecture Settings
    "model": {
        "img_size": 224,
        "patch_size": 4,
        "in_chans": 3,
        "num_classes": 1000,
        "embed_dims": [64, 128, 320, 512],
        "num_heads": [1, 2, 5, 8],
        "num_kv_heads": [1, 1, 1, 2],
        "mlp_ratios": [8, 8, 4, 4],
        "qkv_bias": True,
        "qk_scale": None,
        "drop_rate": 0.0,
        "attn_drop_rate": 0.0,
        "drop_path_rate": 0.2,                         # was 0.1; scaled for 42M params (Swin-T uses 0.2 at 29M)
        "norm_layer": NORM_LAYER,                       # LayerNorm everywhere
        "norm_layer_stage4": NORM_LAYER_STAGE4,         # None = same as stages 1-3
        "depths": [2,2,2,2],
        "sr_ratios": [8, 4, 2, 1],
        "linear": False,
        "use_moe": True,
        # ---- v10 PATCH: MoE block options ----
        "moe_shared_expert": True,    # always-on dense expert alongside the routed ones
        "moe_block_dwconv": True,     # keep PVT v2's DWConv in the MoE'd block
                                      #   (it rides the shared expert -- the routed
                                      #    branch has no token grid to convolve over)
        "moe_capacity_factor": 2.0,
        "moe_gate_noise": 0.5,
        "use_rope": True,
        "rope_last_n_stages": 1,
        "rope_theta": 50,
        "num_stages": 4,
        "moe_last_n_stages": 1,
        "pretrained_path": None,
        "pretrained_hf_id": None,                       # disabled - resuming from our own checkpoint
        "num_frozen_stages": 0,                         # UNFREEZE all stages
        "stage4_lr_multiplier": 10.0,                   # stage 4 gets lr * 10 = 1e-3 (MUST be inside model dict)
    },

    #Train Loop Settings
    "epochs": 100,                                      # half of from-scratch 300
    "lr": 1e-4,                                         # stages 1-3: gentle (they're at epoch 300 maturity)
    "warmup_epochs": 0,
    "start_factor": 1e-6,                               # warmup starts at 1e-6
    "precision": "bf16-mixed",
    "finetuning": False,   # load pretrained weights & reset optimizer/scheduler states for fresh start
    "resuming": True,    # resume entire training (model + optimizer/scheduler states + epoch) from checkpoint

    #Data Settings
    "dataset": "ImageNet",
    # ---- v10 PATCH: micro-batch vs effective batch ----
    # batch_size is what fits in VRAM; accumulate_grad_batches makes up the
    # difference so the OPTIMIZATION still sees `effective_batch_size`, which
    # is what the LR is calibrated for. 1024 = 128 x 8 on a 12 GB card.
    "batch_size": 128,
    "accumulate_grad_batches": 8,
    "effective_batch_size": 1024,      # asserted below; informational
    "num_workers": 8,                  # Windows spawns workers (no fork)

    # ---- v10 PATCH: train a long schedule in pieces ----
    # The cosine is always built for `epochs`. `stop_at_epoch` only ends the
    # run early, so a resumed run continues the SAME schedule rather than
    # restarting a compressed one.
    "milestones": [],                  # e.g. [90, 100, 150, 200]
    "stop_at_epoch": None,             # e.g. 90

    "environment": "local",

    # W&B
    "use_wandb": True,
    "wandb_project": "pvt-moe-imagenet-FINAL",
    "use_tensorboard": True,
}

#-------------- MODEL ARGS-------------#
model_args = {
    "model_config": config["model"],
    "lr": config["lr"],
    "loss_fn": nn.CrossEntropyLoss(),
    "num_classes": config["model"]["num_classes"],
    "task_type": "multiclass",
    "top_k": 1,
    "optimizer_cls": torch.optim.AdamW,
    "optimizer_kwargs": {
        "weight_decay": 5e-2,
        "betas": (0.9, 0.999)
    },
    "scheduler_cls": torch.optim.lr_scheduler.CosineAnnealingLR,
    "scheduler_kwargs": {
        "T_max": config["epochs"],
        "eta_min": 1e-6                                 # was 0; prevents dead final epochs
    },
    "aux_weight": 0.01,                                 # was 0.02; matches V-MoE/NVIDIA/Switch
    "warmup_epochs": config.get("warmup_epochs", 0),
    "start_factor": config.get("start_factor", 0.01),
}

In [ ]:
hf_token  
w_api_key   #None #os.getenv("WANDB_API_KEY")
nb_name= "pvt_tutel_LR_aux_fixed_v7"
# Set your absolute paths again
PROJECT_ROOT = "/workspace/ModelTraining"
DATASET_DIR = "/workspace/ModelTraining/datasets/imagenet_raw/data"

# Replace with your actual token string
#login(token=hf_token)
%env HF_TOKEN=$hf_token
%env WANDB_API_KEY=$w_api_key
%env WANDB_NOTEBOOK_NAME= nb_name

In [ ]:


 ## connected to my private gdrive
if config["environment"] == "native-colab":
  from google.colab import drive
  drive.mount('/content/drive')

  DATASET_PATH = "/content/drive/MyDrive/MS_Thesis/Data"
  CHECKPOINT_PATH = "/content/drive/MyDrive/MS_Thesis/Img_Models"

## temporary paths, deleted on runtime disconnection
elif config["environment"] == "link-collab":
  DATASET_PATH = "/content/data"
  CHECKPOINT_PATH = "/content/trial_model"

elif config["environment"] == "local":
    import os
    
    # 1. Hardcode the exact absolute paths based on your Vast.ai workspace
    PROJECT_ROOT = "/workspace/ModelTraining"
    LOCAL_DATA_ROOT = os.path.join(PROJECT_ROOT, "datasets")
    CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "checkpoints")
    
    # 2. Force HuggingFace to use the specific datasets folder
    os.environ["HF_HOME"] = os.path.join(LOCAL_DATA_ROOT, "hf_cache")
    os.environ["HF_DATASETS_CACHE"] = os.path.join(LOCAL_DATA_ROOT, "hf_cache", "datasets")
    DATASET_PATH = os.path.join(LOCAL_DATA_ROOT, "hf_cache", "datasets")
    
    # 3. Physically build the directories just in case they don't exist yet
    os.makedirs(DATASET_PATH, exist_ok=True)
    os.makedirs(CHECKPOINT_PATH, exist_ok=True)
    
    print(f"[Local] Dataset cache: {DATASET_PATH}")
    print(f"[Local] Checkpoints:   {CHECKPOINT_PATH}")


csv_logger = CSVLogger(save_dir=CHECKPOINT_PATH, name=config["experiment_group"], version=config["version"])

# ── TensorBoard (optional) ────────────────────────────────────────────────────
loggers = [csv_logger]

if config.get("use_tensorboard", False):
    from pytorch_lightning.loggers import TensorBoardLogger
    tb_logger = TensorBoardLogger(
        save_dir=CHECKPOINT_PATH, 
        name=config["experiment_group"], 
        version=config["version"]
    )
    loggers.append(tb_logger)
    tb_dir = f"{CHECKPOINT_PATH}/{config['experiment_group']}/version_{config['version']}"
    print(f"[TensorBoard] Logging to: {tb_dir}")
    print(f"[TensorBoard] View with: tensorboard --logdir {CHECKPOINT_PATH}/{config['experiment_group']}")
else:
    print("[TensorBoard] Disabled (set use_tensorboard=True to enable)")

# ── W&B (optional) ────────────────────────────────────────────────────────────
loggers_wandb = loggers.copy()

if config.get("use_wandb", False):
    import os
    try:
        import wandb
        from pytorch_lightning.loggers import WandbLogger
    except ModuleNotFoundError:
        %pip install --quiet wandb
        import wandb
        from pytorch_lightning.loggers import WandbLogger

    # Try Colab secret first, fall back to interactive login
    api_key = os.getenv("WANDB_API_KEY")
    if api_key:
        wandb.login(key=api_key)
    else:
        print("[W&B] WANDB_API_KEY not found — trying interactive login...")
        wandb.login()

    wandb_logger = WandbLogger(
        project=config.get("wandb_project", "pvt-moe-imagenet"),
        name=config["run_name"],
        config={k: v for k, v in config.items() if k != "model" or not callable(v)},
        save_dir=CHECKPOINT_PATH,
        log_model=False,       # don't upload checkpoints (slow)
    )
    loggers.append(wandb_logger)
    print(f"[W&B] Logging to project: {config.get('wandb_project')}")
else:
    print("[W&B] Disabled (set use_wandb=True to enable)")


In [ ]:
#FOR PRINTING OUTPUTS AT END OF EACH EPOCH
import time

class PrintEpochMetrics(pl.Callback):
    def __init__(self):
        self._epoch_start = None

    def on_train_epoch_start(self, trainer, pl_module):
        self._epoch_start = time.time()

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return

        elapsed = time.time() - self._epoch_start if self._epoch_start else 0
        mins, secs = divmod(elapsed, 60)

        metrics = trainer.callback_metrics
        train_acc  = metrics.get("MulticlassAccuracy/train", torch.tensor(0.0)).item()
        val_acc    = metrics.get("MulticlassAccuracy/val",   torch.tensor(0.0)).item()
        train_loss = metrics.get("Loss/train_epoch", torch.tensor(0.0)).item()
        val_loss   = metrics.get("Loss/val",   torch.tensor(0.0)).item()
        aux_loss   = metrics.get("Aux_loss/train", torch.tensor(0.0)).item()

        print(f"Epoch {trainer.current_epoch:3d} | "
              f"Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Aux: {aux_loss:.4f} | "
              f"Time: {int(mins)}m {int(secs)}s")

    def on_test_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        test_acc  = metrics.get("MulticlassAccuracy/test", torch.tensor(0.0)).item()
        test_loss = metrics.get("Loss/test", torch.tensor(0.0)).item()
        print(f"Test  | Acc: {test_acc:.1%} | Loss: {test_loss:.4f}")


In [ ]:

#-------------- TRAINING ARGS-------------#
trainer_args = {
    "precision" : config.get("precision", "32-true"),
    # v10 PATCH: stop_at_epoch truncates the RUN; the scheduler below is still
    # built for config["epochs"], so the LR trajectory is unchanged.
    "max_epochs" : config.get("stop_at_epoch") or config["epochs"],
    "accelerator": "auto",
    #"fast_dev_run" :True,
    #"limit_train_batches" :2500,
    #"limit_val_batches" : 100,
    "logger": loggers,
    "accumulate_grad_batches" : config.get("accumulate_grad_batches", 1),   # v10 PATCH
    "gradient_clip_val" :5.0,                           # was 1.0; Swin standard, 1.0 clips unnecessarily

    "callbacks" :[
        pl.callbacks.ModelCheckpoint(
            dirpath=CHECKPOINT_PATH + "/" + config["run_name"],
            monitor="MulticlassAccuracy/val",
            mode="max",
            save_weights_only=False,
            save_top_k=2,
            save_last=True,
            # v10 PATCH: the '/' in the old template was interpreted as a path
            # separator, silently creating nested directories per checkpoint.
            auto_insert_metric_name=False,
            filename="epoch{epoch:03d}-valacc{MulticlassAccuracy/val:.4f}",
                                                        # removed every_n_epochs=5 - was missing best checkpoints
        ),
        pl.callbacks.LearningRateMonitor("epoch"),
        # removed EarlyStopping - was killing runs during normal warmup dip (patience=10 too short)
        pl.callbacks.RichProgressBar(),
        PrintEpochMetrics(),
        # v10 PATCH: permanent, never-pruned snapshots you can resume from.
        MilestoneCheckpoint(config.get("milestones", []),
                            CHECKPOINT_PATH + "/" + config["run_name"])]
}

In [ ]:
# ── Prevent CUDA OOM from memory fragmentation at epoch boundaries ──────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = torch.device('cuda' if torch.cuda.is_available() else torch.device('cpu'))

print('Using device:', device)

#Extra things: adding deterministic, benchmark
#torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
# cuda gpu index

torch.set_float32_matmul_precision('high')
pl.seed_everything(42)

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset, DatasetDict

# ── Paths ─────────────────────────────────────────────────────────────────
ARROW_DIR   = "/workspace/ModelTraining/datasets/imagenet_arrow"        # stable Arrow snapshot
PARQUET_DIR = "/workspace/ModelTraining/datasets/imagenet_raw/data/data" # raw parquets (deletable after migration)
CACHE_DIR   = "/workspace/ModelTraining/hf_cache"                       # HF temp cache (deletable after migration)

# ── Transforms ────────────────────────────────────────────────────────────
img_size = config["model"]["img_size"]
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(2, 9),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
    transforms.RandomErasing(p=0.25)
])

val_transforms = transforms.Compose([
    transforms.Resize(int(img_size / 0.875)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# ── Dataset wrapper ───────────────────────────────────────────────────────
class ImageNetDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset   = hf_dataset
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        item  = self.dataset[idx]
        image = item["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, item["label"]

# ── Load dataset ──────────────────────────────────────────────────────────
# Priority:
#   1. load_from_disk(ARROW_DIR)   — instant (~2s), needs nothing else
#   2. load_dataset from HF cache  — needs parquets on disk for fingerprint
#   3. build from scratch           — DISABLED to prevent accidental 160GB rebuild

if os.path.isdir(ARROW_DIR) and os.path.isfile(os.path.join(ARROW_DIR, "train", "dataset_info.json")):
    print(f"⚡ Loading from {ARROW_DIR}")
    raw_dataset = DatasetDict.load_from_disk(ARROW_DIR)

elif os.path.isdir(CACHE_DIR):
    print(f"⚡ Loading directly from HF cache at {CACHE_DIR} ...")
    # This instantly loads the Arrow files HF already built for you!
    raw_dataset = load_dataset(
        "parquet",
        data_files={
            "train":      os.path.join(PARQUET_DIR, "train-*.parquet"),
            "validation": os.path.join(PARQUET_DIR, "validation-*.parquet")
        },
        cache_dir=CACHE_DIR
    )
    
    # WE DELETED THE save_to_disk() LINE HERE TO SAVE 150GB OF SPACE!
    print("✅ Dataset loaded from cache successfully. Skipping Arrow migration to save disk space.")
else:
    raise FileNotFoundError(
        f"No dataset found!\n"
        f"  - No Arrow snapshot at: {ARROW_DIR}\n"
        f"  - No HF cache at: {CACHE_DIR}\n"
        f"To rebuild on a new device, uncomment the block below."
    )

# ── DISABLED: rebuild from scratch (uncomment on a NEW device only) ───────
# os.makedirs(CACHE_DIR, exist_ok=True)
# raw_dataset = load_dataset("parquet", data_files={
#     "train": os.path.join(PARQUET_DIR, "train-*.parquet"),
#     "validation": os.path.join(PARQUET_DIR, "validation-*.parquet")
# }, cache_dir=CACHE_DIR)
# raw_dataset.save_to_disk(ARROW_DIR)

train_ds = ImageNetDataset(raw_dataset["train"],      transform=train_transforms)
val_ds   = ImageNetDataset(raw_dataset["validation"], transform=val_transforms)
print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")


In [ ]:
train_dataloader = DataLoader(
    train_ds,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=config['num_workers'],                        # start low, increase if stable
    pin_memory=True,
    prefetch_factor=2,                    # reduce from 4
    persistent_workers=True,
    drop_last=True,
    multiprocessing_context="fork",       # explicit fork on Linux
)

val_dataloader = DataLoader(
    val_ds,
    batch_size=config["batch_size"] * 2,
    shuffle=False,
    num_workers=config['num_workers'],
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    multiprocessing_context="fork",
)
print("✅ DataLoaders Ready!")
print(f"Train batches: {len(train_dataloader)} | Val batches: {len(val_dataloader)}")

# Test a single batch
batch_imgs, batch_labels = next(iter(train_dataloader))
print(f"Batch -> Images: {batch_imgs.shape} | Labels: {batch_labels.shape}")

In [ ]:
RMSNorm= nn.RMSNorm

In [ ]:
class OverlapPatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, img_size, patch_size, stride, in_chans, embed_dim, norm_layer=nn.LayerNorm):
        super().__init__()

        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)

        assert max(patch_size) > stride, "Set larger patch_size than stride"

        self.img_size = img_size
        self.patch_size = patch_size
        self.H, self.W = img_size[0] // stride, img_size[1] // stride
        self.num_patches = self.H * self.W
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride,
                              padding=(patch_size[0] // 2, patch_size[1] // 2))
        self.norm = norm_layer(embed_dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, RMSNorm)):
            if hasattr(m, "bias") and m.bias is not None: nn.init.constant_(m.bias, 0)
            if hasattr(m, "weight") and m.weight is not None: nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)

        return x, H, W

In [ ]:
class DWConv(nn.Module):
    def __init__(self, dim=768):
        super(DWConv, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1, 2)

        return x

In [ ]:
def _conv_filter(state_dict, patch_size):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    out_dict = {}
    for k, v in state_dict.items():
        if 'patch_embed' in k and 'proj.weight' in k:
            v = v.reshape((v.shape[0], 3, patch_size, patch_size))
        out_dict[k] = v

    return out_dict

In [ ]:
# ============================ v10 PATCH: helpers =============================
# Everything added on top of v9 lives here or is marked "v10 PATCH" inline.
import torch, torch.nn as nn, torch.nn.functional as F
import pytorch_lightning as pl
import os


def build_moe_ffn_layer(
    model_dim: int,
    hidden_size_per_expert: int,
    num_experts: int,
    top_k: int = 1,
    capacity_factor: float = 1.0,
    gate_noise: float = 0.5,
    activation_fn=None,
    group=None,
    **moe_layer_kwargs,
):
    """Wraps tutel_moe.moe_layer, always supplying activation_fn explicitly.

    Tutel's internal default (tutel/experts/ffn.py) does
    `activation_fn = lambda x: F.relu(x)` but the module imports only `torch`
    and `net` -- never `torch.nn.functional as F`. Omitting activation_fn
    therefore raises `NameError: name 'F' is not defined` from inside Tutel's
    FORWARD, not at construction, which makes it look like a model bug.
    Confirmed still present on main at 9a70a681b7673ee23135aa54446ac2a03cf0a61d.
    Passing activation_fn ourselves means that code path never runs.

    NOTE vs the first draft of this helper: capacity_factor and gate_noise must
    be placed INSIDE gate_type. Tutel reads them off the gate config; accepting
    them as plain arguments and not forwarding them silently runs the default
    capacity (1.0) and no gate noise.
    """
    if activation_fn is None:
        activation_fn = lambda x: F.gelu(x)   # matches PVT v2's CFFN activation

    return tutel_moe.moe_layer(
        gate_type={'type': 'top', 'k': top_k,
                   'capacity_factor': capacity_factor,
                   'gate_noise': gate_noise},
        model_dim=model_dim,
        experts={
            'type': 'ffn',
            'count_per_node': num_experts,
            'hidden_size_per_expert': hidden_size_per_expert,
            'activation_fn': activation_fn,
        },
        group=group,
        **moe_layer_kwargs,
    )


class SharedExpertFFN(nn.Module):
    """Always-on dense FFN run alongside the routed experts (DeepSeekMoE style).

    Two reasons it exists:

    1. It is the only branch that can hold a PRETRAINED FFN verbatim. The
       routed experts are built and initialised by Tutel, so a checkpoint can
       only ever be copied into them after the fact.
    2. It can keep PVT v2's DWConv, which the routed branch cannot: token-choice
       routing gathers each expert's tokens out of order and pads them to
       capacity, so there is no H x W grid left to convolve over. Set
       use_dwconv=False for a plain fc1 -> act -> fc2 shared expert.
    """

    def __init__(self, dim, hidden, act_layer=nn.GELU, use_dwconv=True):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.dwconv = DWConv(hidden) if use_dwconv else None
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden, dim)

    def forward(self, x, H, W):
        x = self.fc1(x)
        if self.dwconv is not None:
            x = self.dwconv(x, H, W)
        x = self.act(x)
        return self.fc2(x)


class MilestoneCheckpoint(pl.Callback):
    """Permanent full-state checkpoints at given epoch counts.

    Unlike ModelCheckpoint these are keyed on the epoch COUNT, not a metric,
    and are never pruned by save_top_k -- so an epoch-90 snapshot survives
    another 200 epochs of better validation scores. Each file holds model +
    optimizer + scheduler + epoch, so
    `trainer.fit(..., ckpt_path=<file>)` resumes exactly where it stopped.

    Milestones count COMPLETED epochs: 90 fires once the 90th epoch finishes.
    """

    def __init__(self, milestones, dirpath):
        self.milestones = sorted(set(milestones or []))
        self.dirpath = dirpath

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        completed = trainer.current_epoch + 1
        if completed in self.milestones:
            os.makedirs(self.dirpath, exist_ok=True)
            path = os.path.join(self.dirpath, f"milestone-epoch{completed:03d}.ckpt")
            trainer.save_checkpoint(path)
            print(f"[milestone] epoch {completed}: saved full state -> {path}")


In [ ]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0., linear=False, use_moe=False, num_experts=8,
                 moe_shared_expert=True, moe_block_dwconv=True,          # v10 PATCH
                 moe_capacity_factor=2.0, moe_gate_noise=0.5):           # v10 PATCH
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.use_moe = use_moe
        self.linear = linear
        self.drop = nn.Dropout(drop)

        if self.use_moe:
            # v10 PATCH: routed experts via build_moe_ffn_layer, which always
            # passes activation_fn (Tutel's default branch references an
            # unimported F and raises NameError inside its forward).
            # Also: no .cuda() here -- constructing on the current device and
            # letting Lightning/.to() move it keeps CPU-only work possible.
            self.moe_layer = build_moe_ffn_layer(
                model_dim=in_features,
                hidden_size_per_expert=hidden_features,
                num_experts=num_experts,
                top_k=1,
                capacity_factor=moe_capacity_factor,
                gate_noise=moe_gate_noise,
                activation_fn=act_layer(),
                scan_expert_func=lambda name, param: setattr(param, 'skip_allreduce', True),
                result_func=lambda output: (output, output.l_aux),
            )

            # v10 PATCH: always-on shared expert. Required for function
            # preservation -- see zero_routed_experts() below.
            self.shared_expert = (
                SharedExpertFFN(in_features, hidden_features, act_layer,
                                use_dwconv=moe_block_dwconv)
                if moe_shared_expert else None
            )
            self.zero_routed_experts()
        else:
            self.fc1 = nn.Linear(in_features, hidden_features)
            self.dwconv = DWConv(hidden_features)
            self.act = act_layer()
            self.fc2 = nn.Linear(hidden_features, out_features)

            if self.linear:
              self.relu = nn.ReLU(inplace=True)

        if not self.use_moe:
            self.apply(self._init_weights)

    @torch.no_grad()
    def zero_routed_experts(self):
        """v10 PATCH -- zero every routed expert's fc2 (weight and bias).

        With the shared expert carrying the pretrained FFN, the block then
        computes EXACTLY that FFN at step 0, whatever the untrained router
        decides. The routed experts are not frozen: fc2 still receives
        gradient from the first step (dL/dW2 = h^T g, and h != 0), so they
        learn a residual on top.

        Why not the other way round (clone the FFN into the routed experts and
        zero the shared one)? That is only function-preserving if the combine
        weights are normalised per token, and Tutel normalises gates ONLY when
        top_k > 1 (impls/fast_dispatch.py::extract_critical). At top_k=1 the
        routed branch is scaled by the raw softmax score, so the block would
        emit a FRACTION of the pretrained FFN.
        """
        for name, param in self.moe_layer.named_parameters():
            lname = name.lower()
            if 'gate' in lname or 'wg' in lname:
                continue
            if 'fc2' in lname:
                param.zero_()

    @torch.no_grad()
    def load_from_dense_ffn(self, dense_state):
        """v10 PATCH -- load a pretrained dense FFN into the shared expert.

        `dense_state` maps keys like 'fc1.weight' (any 'mlp.' prefix is
        stripped) to tensors. Routed experts are deliberately left at their
        zeroed fc2. Raises on a shape mismatch rather than half-loading.
        """
        if self.shared_expert is None:
            raise RuntimeError("no shared expert to load into "
                               "(moe_shared_expert=False)")
        target = self.shared_expert.state_dict()
        loaded, skipped = 0, []
        for key, value in dense_state.items():
            name = key.split('mlp.')[-1] if 'mlp.' in key else key
            if name not in target:
                skipped.append(name); continue
            if tuple(target[name].shape) != tuple(value.shape):
                raise ValueError(f"shape mismatch for {name}: checkpoint "
                                 f"{tuple(value.shape)} vs model "
                                 f"{tuple(target[name].shape)}")
            target[name].copy_(value); loaded += 1
        if skipped:
            conv_only = all(k.startswith('dwconv.') for k in skipped)
            why = ("block has no DWConv (moe_block_dwconv=False)"
                   if conv_only and self.shared_expert.dwconv is None
                   else "NO DESTINATION -- check the source")
            print(f"[shared expert] skipped {skipped}: {why}")
        return loaded

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, RMSNorm)):
            if hasattr(m, "bias") and m.bias is not None: nn.init.constant_(m.bias, 0)
            if hasattr(m, "weight") and m.weight is not None: nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
      if self.use_moe:
          b, n, c = x.shape
          x_flat = rearrange(x, 'b n c -> (b n) c').contiguous()
          moe_out = self.moe_layer(x_flat)
          x_out, aux_loss = moe_out
          out = rearrange(x_out, '(b n) c -> b n c', b=b, n=n)
          # v10 PATCH: shared expert sees the ORIGINAL (b, n, c) tensor -- it
          # needs H/W for its DWConv, which the flattened routed path cannot use.
          if self.shared_expert is not None:
              out = out + self.shared_expert(x, H, W)
          return self.drop(out), aux_loss
      else:
        x = self.fc1(x)
        if self.linear:
            x = self.relu(x)
        x = self.dwconv(x, H, W)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

In [ ]:
# ── 2D Rotary Position Embedding (Axial RoPE — complex-mul, from rope-vit) ──

def _init_t_xy(end_x, end_y):
    """Map flattened sequence positions to 2D (x, y) coordinates."""
    t = torch.arange(end_x * end_y, dtype=torch.float32)
    t_x = (t % end_x).float()
    t_y = torch.div(t, end_x, rounding_mode='floor').float()
    return t_x, t_y

def _compute_axial_cis(dim, end_x, end_y, theta=100.0):
    """Build complex frequency cache for axial 2D RoPE (no learnable params)."""
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 4)[: (dim // 4)].float() / dim))
    t_x, t_y = _init_t_xy(end_x, end_y)
    freqs_x = torch.outer(t_x, freqs)
    freqs_y = torch.outer(t_y, freqs)
    freqs_cis = torch.cat([
        torch.polar(torch.ones_like(freqs_x), freqs_x),
        torch.polar(torch.ones_like(freqs_y), freqs_y),
    ], dim=-1)
    return freqs_cis   # (end_x * end_y, dim // 2) complex64

def apply_rotary_emb(x, freqs_cis):
    """Apply axial 2D RoPE via complex multiplication (zero-copy views).
    x: (B, heads, seq, dim) -> rotated (B, heads, seq, dim)"""
    x_ = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs_cis = freqs_cis[None, None].to(x_.device)   # (1, 1, seq, dim//2)
    return torch.view_as_real(x_ * freqs_cis).flatten(-2).type_as(x)

class RotaryEmbedding2D(nn.Module):
    """Cached axial 2D RoPE (rope-vit). No learnable parameters."""
    def __init__(self, head_dim, theta=100.0):
        super().__init__()
        self.head_dim = head_dim
        self.theta = theta
        self._cache = {}

    def get(self, H, W, device):
        key = (H, W, device.index if device.type == 'cuda' else 'cpu')
        if key not in self._cache:
            self._cache[key] = _compute_axial_cis(
                self.head_dim, W, H, self.theta)
        return self._cache[key]

In [ ]:
class GQAttention(nn.Module):
    """SR-Attention with GQA. Optional 2D RoPE via use_rope=True."""
    def __init__(self, dim, num_heads, num_kv_heads, qkv_bias, qk_scale, attn_drop, proj_drop,
                 sr_ratio, linear, norm_layer=nn.LayerNorm, use_rope=False, rope_theta=100.0):
        super().__init__()
        assert dim % num_heads == 0
        assert num_heads % num_kv_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.use_rope = use_rope
        if use_rope:
            assert self.head_dim % 4 == 0, f"head_dim {self.head_dim} must be divisible by 4 for 2D RoPE."
        self.q  = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, 2 * num_kv_heads * self.head_dim, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.linear = linear
        self.sr_ratio = sr_ratio
        if not linear:
            if sr_ratio > 1:
                self.sr   = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
                self.norm = norm_layer(dim)
        else:
            self.pool = nn.AdaptiveAvgPool2d(7)
            self.sr   = nn.Conv2d(dim, dim, kernel_size=1, stride=1)
            self.norm = norm_layer(dim)
            self.act  = nn.GELU()
        if use_rope:
            self.rope = RotaryEmbedding2D(self.head_dim, theta=rope_theta)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, RMSNorm)):
            if hasattr(m, "bias") and m.bias is not None: nn.init.constant_(m.bias, 0)
            if hasattr(m, "weight") and m.weight is not None: nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None: m.bias.data.zero_()

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = rearrange(self.q(x), "b n (h d) -> b h n d", h=self.num_heads, d=self.head_dim)
        if not self.linear:
            if self.sr_ratio > 1:
                x_ = rearrange(x, "b (h w) c -> b c h w", h=H, w=W)
                x_ = self.sr(x_).reshape(B, C, -1).permute(0, 2, 1)
                x_ = self.norm(x_)
                H_kv, W_kv = H // self.sr_ratio, W // self.sr_ratio
            else:
                x_ = x
                H_kv, W_kv = H, W
        else:
            x_ = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_ = self.sr(self.pool(x_)).reshape(B, C, -1).permute(0, 2, 1)
            x_ = self.norm(x_)
            x_ = self.act(x_)
            H_kv, W_kv = 7, 7
        kv = self.kv(x_).reshape(B, -1, 2, self.num_kv_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        if self.use_rope:
            q = apply_rotary_emb(q, self.rope.get(H, W, x.device))
            k = apply_rotary_emb(k, self.rope.get(H_kv, W_kv, x.device))
        x = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0, enable_gqa=True)
        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [ ]:
class Block(nn.Module):

    def __init__(self, dim, num_heads, num_kv_heads, mlp_ratio, qkv_bias, qk_scale, drop, attn_drop,
                 drop_path, act_layer, norm_layer, sr_ratio, linear, use_moe=False, num_experts=8, use_rope=False, rope_theta=100.0,
                 moe_shared_expert=True, moe_block_dwconv=True,          # v10 PATCH
                 moe_capacity_factor=2.0, moe_gate_noise=0.5):           # v10 PATCH
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GQAttention(
            dim,
            num_heads=num_heads, num_kv_heads=num_kv_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale,
            attn_drop=attn_drop, proj_drop=drop, sr_ratio=sr_ratio, linear=linear,
            norm_layer=norm_layer, use_rope=use_rope, rope_theta=rope_theta)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)

        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim,
                       act_layer=act_layer, drop=drop, linear=linear,
                       use_moe=use_moe, num_experts=num_experts,
                       moe_shared_expert=moe_shared_expert,              # v10 PATCH
                       moe_block_dwconv=moe_block_dwconv,                # v10 PATCH
                       moe_capacity_factor=moe_capacity_factor,          # v10 PATCH
                       moe_gate_noise=moe_gate_noise)                    # v10 PATCH

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            if any(hasattr(p, 'skip_allreduce') for p in m.parameters()):
                return
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, RMSNorm)):
            if hasattr(m, "bias") and m.bias is not None: nn.init.constant_(m.bias, 0)
            if hasattr(m, "weight") and m.weight is not None: nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
      x = x + self.drop_path(self.attn(self.norm1(x), H, W))
      mlp_out = self.mlp(self.norm2(x), H, W)
      if isinstance(mlp_out, tuple):
        mlp_out, aux_loss = mlp_out
        x = x + self.drop_path(mlp_out)
        return x, aux_loss
      else:
        x = x + self.drop_path(mlp_out)
        return x

In [ ]:
class PyramidVisionTransformerV2(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, num_classes, embed_dims,
                 num_heads, num_kv_heads, mlp_ratios, qkv_bias, qk_scale, drop_rate,
                 attn_drop_rate, drop_path_rate, norm_layer,
                 depths, sr_ratios, num_stages, linear,stage4_lr_multiplier,
                 use_moe=False, num_experts=8, moe_last_n_stages=1, act_layer=nn.GELU, use_rope=False, rope_last_n_stages=1, rope_theta=100.0,
                 moe_shared_expert=True, moe_block_dwconv=True,          # v10 PATCH
                 moe_capacity_factor=2.0, moe_gate_noise=0.5,            # v10 PATCH
                 norm_layer_stage4=None, ):
        super().__init__()
        self.num_classes = num_classes
        self.depths = depths
        self.num_stages = num_stages
        self.embed_dims = embed_dims

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0

        for i in range(num_stages):
            # Use stage-4-specific norm if provided, else fall back to default
            is_last_n = (i >= num_stages - moe_last_n_stages)
            stage_norm = norm_layer_stage4 if (norm_layer_stage4 and is_last_n) else norm_layer

            patch_embed = OverlapPatchEmbed(img_size=img_size if i == 0 else img_size // (2 ** (i + 1)),
                                            patch_size=7 if i == 0 else 3,
                                            stride=4 if i == 0 else 2,
                                            in_chans=in_chans if i == 0 else embed_dims[i - 1],
                                            embed_dim=embed_dims[i],
                                            norm_layer=stage_norm)

            cutoff_stage = is_last_n
            stage_use_moe = use_moe and cutoff_stage

            rope_cutoff = (i >= num_stages - rope_last_n_stages)
            stage_use_rope = use_rope and rope_cutoff

            block = nn.ModuleList([Block(
                dim=embed_dims[i], num_heads=num_heads[i], num_kv_heads=num_kv_heads[i],
                mlp_ratio=mlp_ratios[i], qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + j], norm_layer=stage_norm,
                sr_ratio=sr_ratios[i], linear=linear, act_layer=act_layer,
                use_moe=stage_use_moe, num_experts=num_experts, use_rope=stage_use_rope, rope_theta=rope_theta,
                moe_shared_expert=moe_shared_expert, moe_block_dwconv=moe_block_dwconv,   # v10 PATCH
                moe_capacity_factor=moe_capacity_factor, moe_gate_noise=moe_gate_noise)   # v10 PATCH
                for j in range(depths[i])])
            norm = stage_norm(embed_dims[i])
            cur += depths[i]

            setattr(self, f"patch_embed{i + 1}", patch_embed)
            setattr(self, f"block{i + 1}", block)
            setattr(self, f"norm{i + 1}", norm)

        self.head = nn.Linear(embed_dims[-1], num_classes) if num_classes > 0 else nn.Identity()
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            if any(hasattr(p, 'skip_allreduce') for p in m.parameters()):
                return
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, RMSNorm)):
            if hasattr(m, "bias") and m.bias is not None: nn.init.constant_(m.bias, 0)
            if hasattr(m, "weight") and m.weight is not None: nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def freeze_patch_emb(self):
        for param in self.patch_embed1.parameters():
            param.requires_grad = False

    @torch.jit.ignore
    def no_weight_decay(self) -> set:
        return set()

    def get_classifier(self):
        return self.head

    def reset_classifier(self, num_classes, global_pool=''):
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dims[-1], num_classes) if num_classes > 0 else nn.Identity()

    def load_pretrained(self, checkpoint_path: str, skip_stage4_mlp: bool = True):
        """Load an official PVT v2 checkpoint."""
        ckpt = torch.load(checkpoint_path, map_location='cpu')
        state_dict = ckpt.get('model', ckpt.get('state_dict', ckpt))

        model_state = self.state_dict()
        filtered = {}
        for k, v in state_dict.items():
            if skip_stage4_mlp and 'block4' in k and 'mlp' in k:
                continue
            if k in model_state and model_state[k].shape != v.shape:
                continue
            filtered[k] = v

        missing, unexpected = self.load_state_dict(filtered, strict=False)
        return missing, unexpected

    def load_pretrained_hf(self, hf_model_id: str = "OpenGVLab/pvt_v2_b1", skip_stage4_mlp: bool = True):
        """Load PVT v2 weights from a HuggingFace model."""
        import re
        from transformers import AutoModelForImageClassification

        hf_model = AutoModelForImageClassification.from_pretrained(hf_model_id)
        hf_state = hf_model.state_dict()

        def remap_key(hf_key):
            # layers.{N}.patch_embedding.* -> patch_embed{N+1}.*
            m = re.match(r'pvt_v2\.encoder\.layers\.(\d+)\.patch_embedding\.(.*)', hf_key)
            if m:
                N, rest = int(m.group(1)), m.group(2)
                rest = re.sub(r'^layer_norm\.', 'norm.', rest)
                rest = re.sub(r'^projection\.', 'proj.', rest)
                return f'patch_embed{N+1}.{rest}'

            # layers.{N}.layer_norm.* -> norm{N+1}.*
            m = re.match(r'pvt_v2\.encoder\.layers\.(\d+)\.layer_norm\.(.*)', hf_key)
            if m:
                N, rest = int(m.group(1)), m.group(2)
                return f'norm{N+1}.{rest}'

            # layers.{N}.blocks.{M}.* -> block{N+1}.{M}.*
            m = re.match(r'pvt_v2\.encoder\.layers\.(\d+)\.blocks\.(\d+)\.(.*)', hf_key)
            if m:
                N, M, rest = int(m.group(1)), int(m.group(2)), m.group(3)
                prefix = f'block{N+1}.{M}'
                rest = re.sub(r'^layer_norm_1\.', 'norm1.', rest)
                rest = re.sub(r'^layer_norm_2\.', 'norm2.', rest)
                rest = re.sub(r'^attention\.query\.', 'attn.q.', rest)
                rest = re.sub(r'^attention\.proj\.', 'attn.proj.', rest)
                rest = re.sub(r'^attention\.spatial_reduction\.', 'attn.sr.', rest)
                rest = re.sub(r'^attention\.layer_norm\.', 'attn.norm.', rest)
                # key/value handled separately below (need concat -> fused kv)
                rest = re.sub(r'^mlp\.dense1\.', 'mlp.fc1.', rest)
                rest = re.sub(r'^mlp\.dense2\.', 'mlp.fc2.', rest)
                return f'{prefix}.{rest}'

            # classifier.* -> head.*
            m = re.match(r'classifier\.(.*)', hf_key)
            if m:
                return f'head.{m.group(1)}'

            return None

        model_state = self.state_dict()
        filtered = {}
        dense_mlp_for_seeding = {}   # v10 PATCH: the MoE'd block's dense FFN
        unmapped = []
        kv_pending = {}

        for hf_key, v in hf_state.items():
            m_kv = re.match(
                r'pvt_v2\.encoder\.layers\.(\d+)\.blocks\.(\d+)\.attention\.(key|value)\.(.*)', hf_key)
            if m_kv:
                N, M, kv_type, suffix = int(m_kv.group(1)), int(m_kv.group(2)), m_kv.group(3), m_kv.group(4)
                block_prefix = f'block{N+1}.{M}'
                pair_key = (block_prefix, suffix)
                kv_pending.setdefault(pair_key, {})[kv_type] = v
                continue

            custom_key = remap_key(hf_key)
            if custom_key is None:
                unmapped.append(hf_key)
                continue
            if skip_stage4_mlp and 'block4' in custom_key and 'mlp' in custom_key:
                # v10 PATCH: keep these instead of discarding them -- they are
                # exactly the dense FFN the MoE'd block replaced, and they seed
                # the shared expert below (sparse upcycling).
                dense_mlp_for_seeding[custom_key] = v
                continue
            if custom_key in model_state and model_state[custom_key].shape != v.shape:
                continue
            filtered[custom_key] = v

        import torch as _torch
        kv_loaded, kv_skipped = 0, 0
        for (block_prefix, suffix), pair in kv_pending.items():
            if 'key' in pair and 'value' in pair:
                fused = _torch.cat([pair['key'], pair['value']], dim=0)
                custom_key = f'{block_prefix}.attn.kv.{suffix}'
                if custom_key in model_state and model_state[custom_key].shape == fused.shape:
                    filtered[custom_key] = fused
                    kv_loaded += 1
                else:
                    kv_skipped += 1

        missing, unexpected = self.load_state_dict(filtered, strict=False)
        print(f"[HF Pretrained] Loaded {len(filtered)} weights from '{hf_model_id}'.")

        # v10 PATCH -- UPCYCLE. Without this the MoE'd block's FFN stays at
        # Tutel's RANDOM init for the whole run: v9 skipped these weights and
        # never put anything in their place, so a "pretrained" MoE run
        # effectively started stage 4 from scratch.
        seeded = 0
        last_blocks = getattr(self, f"block{self.num_stages}", [])
        for idx, blk in enumerate(last_blocks):
            mlp = getattr(blk, 'mlp', None)
            if mlp is None or not getattr(mlp, 'use_moe', False):
                continue
            if getattr(mlp, 'shared_expert', None) is None:
                continue
            prefix = f"block{self.num_stages}.{idx}.mlp."
            dense = {k[len(prefix):]: val
                     for k, val in dense_mlp_for_seeding.items()
                     if k.startswith(prefix)}
            if dense:
                mlp.load_from_dense_ffn(dense)
                mlp.zero_routed_experts()
                seeded += 1
        print(f"  [upcycle] seeded {seeded} MoE block(s): shared expert <- dense "
              f"FFN, routed fc2 zeroed => block output == dense FFN at step 0")
        if kv_loaded or kv_skipped:
            print(f"  KV fused: {kv_loaded} loaded, {kv_skipped} skipped (GQA shape mismatch)")
        if unmapped:
            print(f"  Unmapped HF keys ({len(unmapped)}): {unmapped[:5]}{'...' if len(unmapped) > 5 else ''}")
        if missing:
            print(f"  Missing: {len(missing)} (expected for MoE/GQA)")
            print(f"    {missing[:8]}{'...' if len(missing)>8 else ''}")
        return missing, unexpected

    def freeze_stages(self, num_frozen_stages: int = 3):
        """Freeze the first num_frozen_stages stages."""
        for i in range(num_frozen_stages):
            for attr in [f"patch_embed{i+1}", f"block{i+1}", f"norm{i+1}"]:
                module = getattr(self, attr)
                module.eval()
                for param in module.parameters():
                    param.requires_grad = False

    def forward_features(self, x):
        B = x.shape[0]
        aux_loss_total = 0.0
        num_moe_blocks = 0

        for i in range(self.num_stages):
            patch_embed = getattr(self, f"patch_embed{i + 1}")
            block = getattr(self, f"block{i + 1}")
            norm = getattr(self, f"norm{i + 1}")
            x, H, W = patch_embed(x)
            for blk in block:
                blk_out = blk(x, H, W)
                if isinstance(blk_out, tuple):
                    x, blk_aux = blk_out
                    aux_loss_total = aux_loss_total + blk_aux
                    num_moe_blocks += 1
                else:
                    x = blk_out
            x = norm(x)
            if i != self.num_stages - 1:
                x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        if num_moe_blocks > 0:
            aux_loss_total = aux_loss_total / num_moe_blocks
            return x.mean(dim=1), aux_loss_total
        else:
            return x.mean(dim=1)

    def forward(self, x):
        out = self.forward_features(x)
        if isinstance(out, tuple):
            features, aux_loss = out
            logits = self.head(features)
            return logits, aux_loss
        else:
            features = out
            logits = self.head(features)
            return logits


In [ ]:
import torchmetrics
from torchmetrics import MetricCollection, Accuracy, Precision, Recall, ConfusionMatrix

from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy

In [ ]:

class LitModel(pl.LightningModule):
  def __init__(self, model_config, lr, loss_fn, num_classes, task_type, top_k, scheduler_cls = None, scheduler_kwargs = None, optimizer_cls=None, optimizer_kwargs=None, aux_weight=0.01, warmup_epochs=0, start_factor=0.1):
    super().__init__()
    self.aux_weight = aux_weight
    self.save_hyperparameters(ignore=['loss_fn'])

    #1. Setup model
    model_config = dict(model_config)
    pretrained_path = model_config.pop('pretrained_path', None)
    pretrained_hf_id = model_config.pop('pretrained_hf_id', None)
    num_frozen_stages = model_config.pop('num_frozen_stages', 0)
    self.model = PyramidVisionTransformerV2(**model_config)

    if pretrained_path:
        missing, unexpected = self.model.load_pretrained(pretrained_path)
        print(f"[Pretrained] Loaded '{pretrained_path}'. Missing: {len(missing)}, Unexpected: {len(unexpected)}")
        if missing:
            print(f"  Missing (expected for MoE/GQA): {missing[:8]}{'...' if len(missing)>8 else ''}")
    elif pretrained_hf_id:
        missing, unexpected = self.model.load_pretrained_hf(pretrained_hf_id)
        print(f"[HF Pretrained] Missing: {len(missing)}, Unexpected: {len(unexpected)}")
        if missing:
            print(f"  Missing (expected for MoE/GQA): {missing[:8]}{'...' if len(missing)>8 else ''}")

    if num_frozen_stages > 0:
        self.model.freeze_stages(num_frozen_stages)
        print(f"[Freeze] Froze stages 1\u2013{num_frozen_stages}.")

    #2. Setup Mixup & Cutmix (+label smoothing)
    self.mixup_fn = Mixup(
            mixup_alpha=0.8,
            cutmix_alpha=1.0,
            prob=0.8,
            switch_prob=0.5,
            mode='batch',
            label_smoothing=0.1,
            num_classes=num_classes
        )

    #3. Setup loss functions
    self.train_loss_fn = SoftTargetCrossEntropy()
    self.val_loss_fn = loss_fn

    # 4. Setup Metrics
    metrics = MetricCollection(
        Accuracy(task = task_type, num_classes=num_classes, top_k=top_k),
        Precision(task = task_type, num_classes=num_classes, average='macro', top_k=top_k),
        Recall(task = task_type, num_classes=num_classes, average='macro', top_k=top_k),
    )
    self.train_metrics = metrics.clone(postfix='/train')
    self.val_metrics = metrics.clone(postfix='/val')
    self.test_metrics = metrics.clone(postfix='/test')

    # Standalone ConfusionMatrix - kept out of log_dict (2D tensor can't be log'd as scalar).
    # Logged once at end of training via on_train_end.
    self.val_confmat = ConfusionMatrix(task=task_type, num_classes=num_classes, normalize='true')

  def configure_optimizers(self):
        # -- Discriminative LR: stages 1-3 @ base_lr, stage 4 + head @ higher LR --
        stage4_mult = self.hparams.model_config.get('stage4_lr_multiplier', 1.0)
        base_lr = self.hparams.lr

        stage4_params = set()
        for attr in ['patch_embed4', 'block4', 'norm4', 'head']:
            module = getattr(self.model, attr, None)
            if module is not None:
                for p in module.parameters():
                    if p.requires_grad:
                        stage4_params.add(id(p))

        group_s4, group_rest = [], []
        for p in self.model.parameters():
            if not p.requires_grad:
                continue
            (group_s4 if id(p) in stage4_params else group_rest).append(p)

        param_groups = []
        if group_rest:
            param_groups.append({'params': group_rest, 'lr': base_lr, 'name': 'stages123'})
        if group_s4:
            param_groups.append({'params': group_s4, 'lr': base_lr * stage4_mult, 'name': 'stage4'})

        print(f"[Optimizer] stages 1-3: {len(group_rest)} params @ lr={base_lr}")
        print(f"[Optimizer] stage 4+head: {len(group_s4)} params @ lr={base_lr * stage4_mult}")

        optimizer = self.hparams.optimizer_cls(
            param_groups,
            **self.hparams.optimizer_kwargs
        )

        if self.hparams.scheduler_cls is not None:
            warmup_epochs = getattr(self.hparams, 'warmup_epochs', 0)

            sched_kwargs = dict(self.hparams.scheduler_kwargs)
            if 'T_max' in sched_kwargs and warmup_epochs > 0:
                sched_kwargs['T_max'] = max(1, sched_kwargs['T_max'] - warmup_epochs)

            main_sched = self.hparams.scheduler_cls(
                optimizer=optimizer, **sched_kwargs
            )

            if warmup_epochs > 0:
                warmup_sched = torch.optim.lr_scheduler.LinearLR(
                    optimizer, start_factor=self.hparams.start_factor, total_iters=warmup_epochs
                )
                scheduler = torch.optim.lr_scheduler.SequentialLR(
                    optimizer, [warmup_sched, main_sched],
                    milestones=[warmup_epochs]
                )
            else:
                scheduler = main_sched

            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch",
                    "name": "lr_cosine",
                },
            }
        return optimizer

  def train(self, mode=True):
    super().train(mode)
    if mode:
        # Tutel gate modules override train() and reset to eval — bypass with direct flag
        for m in self.model.modules():
            if hasattr(m, 'gate_noise'):
                m.training = True
    return self

  def on_load_checkpoint(self, checkpoint):
    """Patch LR scheduler T_max when extending training."""
    new_total = self.hparams.scheduler_kwargs.get("T_max", 0)
    warmup = getattr(self.hparams, "warmup_epochs", 0)
    new_cosine_T = max(1, new_total - warmup)
    for ss in checkpoint.get("lr_schedulers", []):
      if "_schedulers" in ss:
        for sub in ss["_schedulers"]:
          if "T_max" in sub: sub["T_max"] = new_cosine_T
      elif "T_max" in ss: ss["T_max"] = new_cosine_T

  def on_train_epoch_end(self):
    """Free fragmented CUDA memory between epochs to prevent OOM at scheduler transitions."""
    import gc
    gc.collect()
    torch.cuda.empty_cache()

  def on_train_epoch_start(self):
    self.model.train()
    # Force tutel gates back to train mode after validation's .eval()
    for name, module in self.model.named_modules():
        if hasattr(module, 'moe_layer'):
            module.moe_layer.train()
            if hasattr(module.moe_layer, 'gates'):
                for g in module.moe_layer.gates:
                    if hasattr(g, 'train'):
                        g.train()
                    g.training = True
    num_frozen = self.hparams.model_config.get('num_frozen_stages', 0)
    if num_frozen > 0:
        self.model.freeze_stages(num_frozen)

  def forward(self, x):
    out = self.model(x)
    if self.training and isinstance(out, tuple):
        logits, aux_loss = out
        return logits, aux_loss
    else:
        return out

  def training_step(self,batch, batch_idx):
    x, y = batch

    if self.mixup_fn is not None:
      x, y = self.mixup_fn(x, y)

    model_out = self.model(x)

    if isinstance(model_out, tuple):
        logits, aux_loss = model_out
        aux_loss = torch.clamp(aux_loss, max=10.0)  # prevent MoE load-balancing spikes → NaN
        ce_loss = self.train_loss_fn(logits, y)
        loss = ce_loss + self.aux_weight * aux_loss
        if torch.isnan(loss) or torch.isinf(loss):
            loss = ce_loss  # drop aux_loss for this step if it produces NaN
    else:
        logits = model_out
        loss = self.train_loss_fn(logits, y)
        aux_loss = torch.tensor(0.0, device=logits.device)
        ce_loss = loss

    self.log('Loss/train_step', loss, on_step=True, on_epoch=False, prog_bar=False)
    self.log('Loss/train_epoch', loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log('CE_loss/train', ce_loss, on_step=False, on_epoch=True)
    self.log('Aux_loss/train', aux_loss, on_step=False, on_epoch=True, prog_bar=True)

    y_hard = y.argmax(dim=1)
    output = self.train_metrics(logits, y_hard)
    self.log_dict(output, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def on_validation_epoch_start(self):
    # v10 PATCH: torchmetrics objects updated by hand are NOT auto-reset by
    # Lightning (only self.log-routed ones are). Without this the "final"
    # confusion matrix aggregated EVERY epoch of the run plus the sanity-check
    # batches, so it never showed the final model's behaviour.
    self.val_confmat.reset()

  def validation_step(self,batch, batch_idx):
    x, y = batch
    model_out = self.model(x)
    if isinstance(model_out, tuple):
      logits, _ = model_out
    else:
      logits = model_out

    loss = self.val_loss_fn(logits, y)
    self.log('Loss/val', loss, on_epoch=True, prog_bar=True)

    output = self.val_metrics(logits, y)
    self.log_dict(output, on_epoch=True, prog_bar=True)

    self.val_confmat.update(logits, y)

  def on_train_end(self):
    """Log the final validation confusion matrix once at the end of training."""
    if not self.val_confmat._update_called:
        return
    cm = self.val_confmat.compute().cpu().numpy()
    try:
        import matplotlib.pyplot as plt
        import wandb
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(cm, cmap='Blues', aspect='auto')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title(f'Val Confusion Matrix (epoch {self.current_epoch}, row-normalized)')
        for lg in (self.loggers or []):
            exp = getattr(lg, 'experiment', None)
            if exp is not None and isinstance(exp, wandb.sdk.wandb_run.Run):
                exp.log({"val/confmat_final": wandb.Image(fig)})
                break
        plt.close(fig)
    except Exception as e:
        print(f"[confmat] skipped logging: {e}")

  def test_step(self, batch, batch_idx):
    x, y = batch
    model_out = self.model(x)
    if isinstance(model_out, tuple):
        logits, _ = model_out
    else:
        logits = model_out

    loss = self.val_loss_fn(logits, y)
    self.log('Loss/test', loss, on_epoch=True, prog_bar=True)

    output = self.test_metrics(logits, y)
    self.log_dict(output, on_epoch=True, prog_bar=True)

In [ ]:
pl.seed_everything(42)

if config["finetuning"] and not config["resuming"]:
    # ── Set checkpoint for FINE-TUNE (weights only; fresh optimizer/scheduler/epoch)
    RESUME_CKPT = None  
    FINE_TUNE_CKPT = config['ckpt_path']  
    
    # FIX 1 & 2: Removed quotes around the variable and added strict=False
    model = LitModel.load_from_checkpoint(FINE_TUNE_CKPT, strict=False, weights_only=False, **model_args)
    print(f"✅ Loaded Phase 1 weights from {FINE_TUNE_CKPT} for Fine-Tuning!")

elif not config["finetuning"] and config["resuming"]:
    # ── REUSE previous optimizer state and epoch count
    RESUME_CKPT = config['ckpt_path']
    model = LitModel(**model_args)
    print(f"✅ Resuming exact training state from {RESUME_CKPT}")

else:
    # ── Start totally fresh (no checkpoint)
    RESUME_CKPT = None
    model = LitModel(**model_args)
    print("✅ Starting completely fresh training run.")
'''
RESUME_CKPT="/workspace/ModelTraining/checkpoints/pvt_tutel_LR_aux_fixed_v7/last.ckpt"
model = LitModel(**model_args)
'''
# Force tutel MoE gate into train mode
for name, m in model.named_modules():
    if hasattr(m, 'gate_noise'):
        m.train()
        print(f"[Fix] Forced {name} to train mode")

In [ ]:
model.model.to(device)

# Force tutel MoE gates into train mode (they don't respond to .train() properly)
for name, module in model.named_modules():
    if hasattr(module, 'moe_layer'):
        moe = module.moe_layer
        moe.train()
        # Tutel stores gates in a .gates list — force each one
        if hasattr(moe, 'gates'):
            for g in moe.gates:
                if hasattr(g, 'train'):
                    g.train()
                g.training = True
        # Also check _moe_layer attribute (some tutel versions)
        print(f"[MoE Fix] {name}.moe_layer -> training={moe.training}")

In [ ]:
trainer = pl.Trainer(**trainer_args)

# ckpt_path=None always: fine-tuning uses fresh optimizer/scheduler.
# Weights (if any) were loaded in Cell 66 via LitModel.load_from_checkpoint.
# v10 PATCH: `weights_only` is NOT a Trainer.fit argument -- the old call
# raised TypeError before training ever started.
trainer.fit(model, train_dataloader, val_dataloader, ckpt_path=RESUME_CKPT)

print(f"[done] best checkpoint: {trainer.checkpoint_callback.best_model_path}")
print("Resume this schedule later (here or on another machine) with:")
print(f"  RESUME_CKPT = '{CHECKPOINT_PATH}/{config['run_name']}/milestone-epochNNN.ckpt'")

In [ ]:
# Find last & best checkpoint paths
print("Last:", trainer.checkpoint_callback.last_model_path)
print("Best:", trainer.checkpoint_callback.best_model_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# ── Read CSV logs ──────────────────────────────────────────────────────────────
log_dir = os.path.join(CHECKPOINT_PATH, config["experiment_group"], f"version_{config['version']}")
metrics_path = os.path.join(log_dir, "metrics.csv")

df = pd.read_csv(metrics_path)
print(f"Columns: {list(df.columns)}")

# ── Helper: find column by keywords ───────────────────────────────────────────
def find_col(df, *keywords):
    for c in df.columns:
        if all(k in c.lower() for k in keywords):
            return c
    return None

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Train/Val Accuracy ────────────────────────────────────────────────
train_acc_col = find_col(df, 'accuracy', 'train') or find_col(df, 'acc', 'train')
val_acc_col   = find_col(df, 'accuracy', 'val')   or find_col(df, 'acc', 'val')

if train_acc_col:
    acc_train = df[['epoch', train_acc_col]].dropna()
    axes[0].plot(acc_train['epoch'], acc_train[train_acc_col], label='Train')
if val_acc_col:
    acc_val = df[['epoch', val_acc_col]].dropna()
    axes[0].plot(acc_val['epoch'], acc_val[val_acc_col], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Train / Val Accuracy'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Plot 2: Train/Val Loss ───────────────────────────────────────────────────
train_loss_col = find_col(df, 'loss', 'train')
val_loss_col   = find_col(df, 'loss', 'val')

if train_loss_col:
    loss_train = df[['epoch', train_loss_col]].dropna()
    axes[1].plot(loss_train['epoch'], loss_train[train_loss_col], label='Train')
if val_loss_col:
    loss_val = df[['epoch', val_loss_col]].dropna()
    axes[1].plot(loss_val['epoch'], loss_val[val_loss_col], label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Train / Val Loss'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ── Plot 3: Aux Loss (MoE load balancing) ────────────────────────────────────
aux_col = find_col(df, 'aux')
if aux_col:
    aux = df[['epoch', aux_col]].dropna()
    axes[2].plot(aux['epoch'], aux[aux_col], label='Aux Loss', color='orange')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Aux Loss')
    axes[2].set_title('MoE Aux Loss (load balancing)'); axes[2].legend()
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'No aux loss column found', ha='center', va='center')

plt.tight_layout()
plt.savefig(os.path.join(log_dir, "training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {log_dir}/training_curves.png")


In [ ]:
# ── MoE Expert Utilization Diagnostic ──────────────────────────────────────────
# Runs a few val batches and visualizes how tokens are routed across experts.

import matplotlib.pyplot as plt
import torch
import numpy as np

@torch.no_grad()
def diagnose_expert_utilization(model, dataloader, num_batches=50, device='cuda'):
    """Collect gate routing decisions from all MoE blocks."""
    model.eval()
    model.to(device)

    # Find all MoE Mlp modules
    moe_modules = []
    for name, module in model.named_modules():
        if hasattr(module, 'use_moe') and module.use_moe and hasattr(module, 'moe_layer'):
            moe_modules.append((name, module))

    if not moe_modules:
        print("No MoE modules found.")
        return

    num_experts = moe_modules[0][1].moe_layer.num_global_experts
    # Accumulate per-module expert counts
    all_counts = {name: torch.zeros(num_experts, dtype=torch.long) for name, _ in moe_modules}

    for batch_idx, (x, _) in enumerate(dataloader):
        if batch_idx >= num_batches:
            break
        x = x.to(device)
        B = x.shape[0]

        # Run through each stage manually to capture gate decisions
        for i in range(model.num_stages):
            patch_embed = getattr(model, f"patch_embed{i+1}")
            block_list  = getattr(model, f"block{i+1}")
            norm        = getattr(model, f"norm{i+1}")
            x_stage, H, W = patch_embed(x)

            for blk in block_list:
                x_stage = x_stage + blk.attn(blk.norm1(x_stage), H, W)
                mlp_input = blk.norm2(x_stage)

                if blk.mlp.use_moe:
                    b, n, c = mlp_input.shape
                    x_flat = mlp_input.reshape(b * n, c)

                    # Get gate logits from Tutel's gate
                    gate = blk.mlp.moe_layer.gates[0]
                    gate_logits = gate.wg(x_flat)  # [B*N, num_experts]
                    expert_idx = gate_logits.argmax(dim=-1)  # [B*N]

                    # Find module name
                    for mn, mm in moe_modules:
                        if mm is blk.mlp:
                            all_counts[mn] += torch.bincount(
                                expert_idx.cpu(), minlength=num_experts
                            )
                            break

                # Run actual forward for next stage input
                blk_out = blk.mlp(mlp_input, H, W)
                if isinstance(blk_out, tuple):
                    mlp_out, _ = blk_out
                else:
                    mlp_out = blk_out
                x_stage = x_stage + mlp_out

            x_stage = norm(x_stage)
            if i != model.num_stages - 1:
                x_stage = x_stage.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
            x = x_stage

    # ── Plot ──────────────────────────────────────────────────────────────────
    n_modules = len(moe_modules)
    fig, axes = plt.subplots(1, n_modules, figsize=(7 * n_modules, 5))
    if n_modules == 1:
        axes = [axes]

    for ax, (name, _) in zip(axes, moe_modules):
        counts = all_counts[name].float()
        total = counts.sum()
        pcts = (counts / total * 100).numpy()
        ideal = 100.0 / num_experts

        bars = ax.bar(range(num_experts), pcts, color='steelblue', edgecolor='black', alpha=0.8)
        ax.axhline(y=ideal, color='red', linestyle='--', label=f'Ideal ({ideal:.1f}%)')

        # Color bars by deviation from ideal
        for bar, pct in zip(bars, pcts):
            if pct < ideal * 0.5:
                bar.set_color('tomato')       # underloaded (possible collapse)
            elif pct > ideal * 1.5:
                bar.set_color('gold')         # overloaded

        ax.set_xlabel('Expert ID')
        ax.set_ylabel('Token share (%)')
        ax.set_title(f'{name}\n({total:.0f} tokens)')
        ax.legend()
        ax.set_xticks(range(num_experts))
        ax.grid(True, alpha=0.3, axis='y')

        # Entropy
        probs = counts / total
        probs = probs[probs > 0]
        entropy = -(probs * probs.log()).sum().item()
        max_entropy = np.log(num_experts)
        ax.text(0.02, 0.95, f'Entropy: {entropy:.2f} / {max_entropy:.2f} (max)',
                transform=ax.transAxes, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle('MoE Expert Utilization', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Print summary
    for name, _ in moe_modules:
        counts = all_counts[name].float()
        total = counts.sum()
        pcts = counts / total * 100
        print(f"\n{name}:")
        print(f"  Tokens: {total:.0f}")
        for i, p in enumerate(pcts):
            bar = '█' * int(p * 2)
            print(f"  Expert {i}: {p:5.1f}% {bar}")

# ── Run diagnostic ────────────────────────────────────────────────────────────
diagnose_expert_utilization(model.model, val_dataloader, num_batches=50)


In [ ]:

# 1. Load the latest version of the logs
# Adjust 'version_0' if you have multiple runs (e.g., version_1, version_2)
log_path = os.path.join(CHECKPOINT_PATH, config['experiment_group'], f"version_{config['version']}", "metrics.csv")

if not os.path.exists(log_path):
    print(f"Error: Could not find log file at {log_path}")
    print("Did you enable CSVLogger in the Trainer?")
else:
    # 2. Load and Group Data
    metrics = pd.read_csv(log_path)

    # Group by epoch to merge the separate train/val rows
    epoch_metrics = metrics.groupby('epoch').mean()

    # 3. Automatic Column Detection (Finds 'acc', 'Accuracy', 'loss', etc.)
    # This ensures it works whether you used 'val_MulticlassAccuracy' or 'Accuracy/val'
    acc_cols = [c for c in epoch_metrics.columns if 'acc' in c.lower() or 'accuracy' in c.lower()]
    loss_cols = [c for c in epoch_metrics.columns if 'loss' in c.lower()]

    # Separate Train vs Val
    train_acc_col = [c for c in acc_cols if 'train' in c.lower()][0]
    val_acc_col   = [c for c in acc_cols if 'val' in c.lower()][0]

    train_loss_col = [c for c in loss_cols if 'train' in c.lower()][0]
    val_loss_col   = [c for c in loss_cols if 'val' in c.lower()][0]

    # 4. Plotting
    plt.figure(figsize=(12, 5))

    # --- Plot Accuracy ---
    plt.subplot(1, 2, 1)
    plt.plot(epoch_metrics.index, epoch_metrics[train_acc_col], label='Train Acc', marker='o')
    plt.plot(epoch_metrics.index, epoch_metrics[val_acc_col], label='Val Acc', marker='o')
    plt.title(f'Accuracy (Best: {epoch_metrics[val_acc_col].max():.1%})')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # --- Plot Loss ---
    plt.subplot(1, 2, 2)
    # Plot per-step loss (many points per epoch, smoother curve)
    if 'Loss/train_step' in epoch_metrics.columns:
        plt.plot(epoch_metrics.index, epoch_metrics['Loss/train_step'], label='Train Loss (per-step)', alpha=0.6, linewidth=1)
    # Plot per-epoch loss (one point per epoch, aggregated)
    if 'Loss/train_epoch' in epoch_metrics.columns:
        plt.plot(epoch_metrics.index, epoch_metrics['Loss/train_epoch'], label='Train Loss (per-epoch)', marker='o', linewidth=2)
    # Fallback to old column name if neither exists
    if train_loss_col and train_loss_col not in ['Loss/train_step', 'Loss/train_epoch']:
        plt.plot(epoch_metrics.index, epoch_metrics[train_loss_col], label='Train Loss', marker='o')
    plt.plot(epoch_metrics.index, epoch_metrics[val_loss_col], label='Val Loss', marker='o')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
from fvcore.nn import FlopCountAnalysis, flop_count_table
import torch

def count_flops(model, img_size=224):
    torch.no_grad() #model.eval()
    x = torch.randn(1, 3, img_size, img_size, device=next(model.parameters()).device)

    # Patch MoE layers to equivalent dense FFN for counting
    # Top-1 MoE effective FLOPs = 1 expert's FFN (not 8)
    moe_flops = 0
    for name, module in model.named_modules():
        if hasattr(module, 'moe_layer'):
            # stage 4 FFN: fc1 (dim→hidden) + fc2 (hidden→dim), per token
            # stage 4: seq_len = (img_size/32)^2 = 49, dim=512, hidden=512*4=2048
            dim     = module.moe_layer.model_dim
            hidden  = module.moe_layer.experts.hidden_size_per_expert
            seq_len = (img_size // 32) ** 2          # stage 4 patch grid
            # 2× for multiply-add, ×2 for fc1+fc2
            moe_flops += 2 * seq_len * dim * hidden  # fc1
            moe_flops += 2 * seq_len * hidden * dim  # fc2

    # Count non-MoE ops with fvcore
    # Temporarily replace MoE forward to avoid tracing errors
    original_forwards = {}
    for name, module in model.named_modules():
        if hasattr(module, 'moe_layer'):
            original_forwards[name] = module.forward
            # stub: return zeros of correct shape + 0 aux loss
            def _stub(self, x, H, W, _mod=module):
                b, n, c = x.shape
                return torch.zeros_like(x), torch.tensor(0.0)
            import types
            module.forward = types.MethodType(_stub, module)

    with torch.no_grad():
        flops = FlopCountAnalysis(model, x)
        flops.unsupported_ops_warnings(False)
        flops.uncalled_modules_warnings(False)
        base_flops = flops.total()

    # Restore
    for name, module in model.named_modules():
        if name in original_forwards:
            module.forward = original_forwards[name]

    total_flops = base_flops + moe_flops
    print(f"Non-MoE FLOPs:  {base_flops/1e9:.3f} GFLOPs")
    print(f"MoE FLOPs:      {moe_flops/1e9:.3f} GFLOPs  (top-1, 1 expert active)")
    print(f"Total:          {total_flops/1e9:.3f} GFLOPs")
    print()
    print(flop_count_table(flops, max_depth=3))
    return total_flops

count_flops(model.model)


In [ ]:
###IN CASE NEED TO CLEAR CHECKPOINTS###
'''
%%bash
KEEP="pvitb1_imagenet_satt_tutelmoe_FINAL_FTs4RUN"
BASE="/workspace/ModelTraining/checkpoints"

for dir in "$BASE"/*/; do
    name=$(basename "$dir")
    # Keep the frozen run AND wandb directory
    if [ "$name" != "$KEEP" ] && [ "$name" != "wandb" ]; then
        echo "Deleting: $dir"
        rm -rf "$dir"
    fi
done
echo "Done. Kept: $KEEP and wandb/"
'''